# Cover-2 and Liquidity Stress Lab

This lab evaluates how Cover-2 adequacy can degrade when we add:

- liquidity close-out costs,
- higher concentration penalties,
- tighter survivor liquidity buffers,
- assessment-driven second-round stress.

Outputs are generated as reproducible report bundles in `reports/`:

- `report.md`
- `scenario_comparison.csv`
- `scenario_comparison_summary.json`

## 1) Setup

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"
REPORTS_DIR = REPO_ROOT / "reports"
CONFIG_DIR = REPORTS_DIR / "cover2_liquidity_lab_configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ENV = dict(os.environ)
ENV["PYTHONPATH"] = f"{SRC_DIR}:{ENV.get('PYTHONPATH', '')}" if ENV.get("PYTHONPATH") else str(SRC_DIR)

print("Repo:", REPO_ROOT)
print("Config dir:", CONFIG_DIR)

## 2) Config Builder

We keep margin method fixed (`stressed_var`) and vary liquidity/contagion severity so we can isolate Cover-2 degradation.

In [ ]:
def cover2_config(
    *,
    run_name: str,
    volatility: float,
    liquidity_mult: float,
    include_closeout: bool,
    base_spread_cost: float,
    vol_liq_multiplier: float,
    concentration_penalty: float,
    market_depth_penalty: float,
    liquidity_breach_ratio: float,
    survivor_liquidity_scale: float,
):
    return {
        "run_name": run_name,
        "time_horizon_days": 252,
        "include_liquidity_closeout_cost": include_closeout,
        "margin": {
            "method": "stressed_var",
            "confidence_level": 0.99,
            "margin_period_of_risk_days": 5,
            "stressed_var_vol_multiplier": 1.6,
            "anti_procyclicality_buffer": 0.05,
            "volatility_floor": 0.12,
            "concentration_addon": 0.04,
            "liquidity_addon": 0.06,
        },
        "waterfall": {
            "ccp_skin_in_game": 80000000,
            "assessment_cap_multiple": 0.20,
            "include_defaulter_df": True,
            "default_fund_method": "cover2",
            "default_fund_value": 0.0,
        },
        "closeout": {
            "base_spread_cost": base_spread_cost,
            "volatility_liquidity_multiplier": vol_liq_multiplier,
            "concentration_penalty": concentration_penalty,
            "market_depth_penalty": market_depth_penalty,
        },
        "contagion": {
            "liquidity_breach_ratio": liquidity_breach_ratio,
        },
        "scenarios": [
            {
                "scenario_name": "cover2_liquidity_stress",
                "stress_label": "liquidity_stress",
                "volatility": volatility,
                "liquidity_spread_multiplier": liquidity_mult,
                "jump_probability": 0.03,
                "jump_size_distribution": "normal(mean=-0.10,std=0.05)",
            }
        ],
        "members": [
            {
                "member_id": "CM_DEF",
                "default_fund_contribution": 90000,
                "liquidity_buffer": int(220000 * survivor_liquidity_scale),
                "portfolio": {"positions": [{"asset_id": "IDX", "quantity": 1800000, "price": 1.0}]},
            },
            {
                "member_id": "CM_SURV_1",
                "default_fund_contribution": 150000,
                "liquidity_buffer": int(260000 * survivor_liquidity_scale),
                "portfolio": {"positions": [{"asset_id": "IDX", "quantity": 1100000, "price": 1.0}]},
            },
            {
                "member_id": "CM_SURV_2",
                "default_fund_contribution": 140000,
                "liquidity_buffer": int(250000 * survivor_liquidity_scale),
                "portfolio": {"positions": [{"asset_id": "IDX", "quantity": 1000000, "price": 1.0}]},
            },
            {
                "member_id": "CM_SURV_3",
                "default_fund_contribution": 120000,
                "liquidity_buffer": int(230000 * survivor_liquidity_scale),
                "portfolio": {"positions": [{"asset_id": "IDX", "quantity": 900000, "price": 1.0}]},
            },
        ],
    }

## 3) Build Baseline and Stress Configs

In [ ]:
baseline = cover2_config(
    run_name="cover2_baseline",
    volatility=0.24,
    liquidity_mult=1.0,
    include_closeout=False,
    base_spread_cost=0.0,
    vol_liq_multiplier=0.0,
    concentration_penalty=0.0,
    market_depth_penalty=0.0,
    liquidity_breach_ratio=1.0,
    survivor_liquidity_scale=1.0,
)

liquidity_stress = cover2_config(
    run_name="cover2_liquidity_stress",
    volatility=0.38,
    liquidity_mult=1.5,
    include_closeout=True,
    base_spread_cost=0.01,
    vol_liq_multiplier=0.025,
    concentration_penalty=0.08,
    market_depth_penalty=0.015,
    liquidity_breach_ratio=1.0,
    survivor_liquidity_scale=0.8,
)

assessment_contagion_stress = cover2_config(
    run_name="cover2_assessment_contagion_stress",
    volatility=0.42,
    liquidity_mult=1.6,
    include_closeout=True,
    base_spread_cost=0.012,
    vol_liq_multiplier=0.03,
    concentration_penalty=0.10,
    market_depth_penalty=0.02,
    liquidity_breach_ratio=0.85,
    survivor_liquidity_scale=0.65,
)

baseline_path = CONFIG_DIR / "cover2_baseline.json"
liquidity_stress_path = CONFIG_DIR / "cover2_liquidity_stress.json"
contagion_stress_path = CONFIG_DIR / "cover2_assessment_contagion_stress.json"

baseline_path.write_text(json.dumps(baseline, indent=2), encoding="utf-8")
liquidity_stress_path.write_text(json.dumps(liquidity_stress, indent=2), encoding="utf-8")
contagion_stress_path.write_text(json.dumps(assessment_contagion_stress, indent=2), encoding="utf-8")

print("Wrote configs:")
print("-", baseline_path)
print("-", liquidity_stress_path)
print("-", contagion_stress_path)

## 4) Bundle Runner Helper

In [ ]:
def run_bundle(stressed_config: Path, baseline_config: Path, bundle_name: str) -> Path:
    bundle_dir = REPORTS_DIR / bundle_name
    cmd = [
        sys.executable,
        "-m",
        "clearrisk.cli",
        "report",
        "--config",
        str(stressed_config),
        "--compare-config",
        str(baseline_config),
        "--bundle-dir",
        str(bundle_dir),
    ]
    completed = subprocess.run(cmd, env=ENV, check=True, capture_output=True, text=True)
    print(completed.stdout)
    return bundle_dir

## 5) Generate Reproducible Bundles

In [ ]:
bundles = {}
bundles["baseline_vs_liquidity_stress"] = run_bundle(
    liquidity_stress_path,
    baseline_path,
    "lab_cover2_baseline_vs_liquidity_stress",
)

bundles["baseline_vs_contagion_stress"] = run_bundle(
    contagion_stress_path,
    baseline_path,
    "lab_cover2_baseline_vs_contagion_stress",
)

print("Bundle paths:")
for k, v in bundles.items():
    print(f"- {k}: {v}")

## 6) Concise Interpretation Tables

We extract compact interpretation-ready fields from each bundle JSON summary.

In [ ]:
def load_summary(bundle_dir: Path):
    summary_path = bundle_dir / "scenario_comparison_summary.json"
    return json.loads(summary_path.read_text(encoding="utf-8"))

rows = []
for experiment, bundle_dir in bundles.items():
    s = load_summary(bundle_dir)
    c = s.get("comparison", {})
    rows.append(
        {
            "experiment": experiment,
            "tail_loss_ratio_es": c.get("tail_loss_ratio_es", 0.0),
            "margin_adequacy_delta": c.get("margin_adequacy_delta", 0.0),
            "cover2_adequacy_delta": c.get("cover2_adequacy_delta", 0.0),
            "cover2_adequacy_ratio": c.get("cover2_adequacy_ratio", 0.0),
        }
    )

summary_table = pd.DataFrame(rows)
summary_table

In [ ]:
summary_table.style.format(
    {
        "tail_loss_ratio_es": "{:.3f}",
        "margin_adequacy_delta": "{:.3f}",
        "cover2_adequacy_delta": "{:.3f}",
        "cover2_adequacy_ratio": "{:.3f}",
    }
)

## 7) Practitioner CSV Preview (One Bundle)

In [ ]:
csv_preview_path = bundles["baseline_vs_contagion_stress"] / "scenario_comparison.csv"
pd.read_csv(csv_preview_path)

## 8) Suggested Interpretation Template

- If `tail_loss_ratio_es` rises materially, stressed tail losses are outpacing baseline assumptions.
- If `cover2_adequacy_delta` is negative, Cover-2 resilience deteriorates under stress extensions.
- If `margin_adequacy_delta` is negative while close-out costs are on, liquidity-adjusted losses are dominating margin protections.

Use this notebook for educational scenario-design and portfolio demonstration, not production decisioning.